# VitaGraph — Worked Examples

Notebook counterpart to the scripts in `examples/`. Each section is
self-contained and mirrors one example script.

## Example 1 — Generate and export a dataset (`examples/generate_dataset.py`)

In [1]:
from pathlib import Path

from vitagraph import SyntheticCohortGenerator

Path("outputs").mkdir(exist_ok=True)
cohort = SyntheticCohortGenerator(seed=7).generate(num_samples=300)
cohort.to_csv("outputs/notebook_cohort.csv", index=False)
cohort.sample(5, random_state=1)

,chronological_age,heart_rate_avg,hrv_avg,sleep_hours_avg,activity_level,environmental_exposure,biological_age
189,36.0,78.650861,67.543125,8.692074,0.662784,0.681403,36.664094
123,51.0,83.775773,53.232944,8.816690,0.481321,0.149373,53.765797
185,43.0,60.024190,43.400113,7.403110,0.872384,0.021342,40.650475
213,49.0,74.735778,49.729923,5.545789,0.458990,0.466521,59.921564
106,53.0,62.474401,35.024780,7.720180,0.619225,0.000000,49.477124


## Example 2 — Train & compare models (`examples/train_model.py`)

In [2]:
from sklearn.model_selection import train_test_split

from vitagraph import BioAgeEstimator
from vitagraph.synthetic_data import FEATURE_COLUMNS, TARGET_COLUMN

X, y = cohort[FEATURE_COLUMNS], cohort[TARGET_COLUMN]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=7)

best, best_mae = None, float("inf")
for model_type in ["linear", "random_forest", "gradient_boosting"]:
    est = BioAgeEstimator(model_type=model_type).train(X_train, y_train)
    mae = est.evaluate(X_test, y_test).mae
    print(f"{model_type:<20} MAE={mae:.2f}")
    if mae < best_mae:
        best, best_mae = est, mae

print(f"\nBest: {best.model_type} (MAE={best_mae:.2f})")
best.save_model("models/notebook_demo.joblib")

2026-07-18 13:23:08 | INFO     | vitagraph.bio_age_estimator | Training linear on 225 samples (6 features)


2026-07-18 13:23:08 | INFO     | vitagraph.bio_age_estimator | Evaluation — MAE: 1.65  RMSE: 2.02  R²: 0.981


linear               MAE=1.65
2026-07-18 13:23:08 | INFO     | vitagraph.bio_age_estimator | Training random_forest on 225 samples (6 features)


2026-07-18 13:23:08 | INFO     | vitagraph.bio_age_estimator | Evaluation — MAE: 2.12  RMSE: 2.61  R²: 0.969


random_forest        MAE=2.12
2026-07-18 13:23:08 | INFO     | vitagraph.bio_age_estimator | Training gradient_boosting on 225 samples (6 features)


2026-07-18 13:23:09 | INFO     | vitagraph.bio_age_estimator | Evaluation — MAE: 2.04  RMSE: 2.63  R²: 0.968


gradient_boosting    MAE=2.04

Best: linear (MAE=1.65)
2026-07-18 13:23:09 | INFO     | vitagraph.bio_age_estimator | Saved model to models/notebook_demo.joblib (+ metadata at models/notebook_demo.json)


PosixPath('models/notebook_demo.joblib')

## Example 3 — Build & visualize a graph (`examples/visualize_graph.py`)

In [3]:
from datetime import datetime

from vitagraph import BioSignalProcessor, KnowledgeGraph
from vitagraph.visualization import plot_person_subgraph

processor = BioSignalProcessor(seed=7)
start = datetime(2026, 2, 1)
hr_df = processor.generate_synthetic_heart_rate(30, start, interval_minutes=60)
sleep_df = processor.generate_synthetic_sleep_data(10, start.date())

kg = KnowledgeGraph()
kg.build_from_processed_data("P042", hr_df, sleep_df)
print(kg.get_graph_info())
plot_person_subgraph(kg, "P042", "outputs/notebook_P042.png")

{'nodes': 41, 'edges': 40, 'node_type_counts': {'Person': 1, 'BiometricData': 40}, 'edge_relations': ['HAS_DATA']}


2026-07-18 13:23:09 | INFO     | vitagraph.visualization | Saved subgraph plot for P042 to outputs/notebook_P042.png


PosixPath('outputs/notebook_P042.png')

## Example 4 — Full pipeline via the library API (`examples/cli_demo.py` equivalent)

In [4]:
from vitagraph.config import PipelineDefaults
from vitagraph.pipeline import run_pipeline

result = run_pipeline(config=PipelineDefaults(num_individuals=3, training_samples=300), output_dir="outputs/notebook_pipeline")
result.individuals

2026-07-18 13:23:09 | INFO     | vitagraph.pipeline | Starting VitaGraph pipeline (model=gradient_boosting, individuals=3, seed=42)


2026-07-18 13:23:09 | INFO     | vitagraph.bio_signal_processor | Processing 'heart_rate' (300 rows, window=5)


2026-07-18 13:23:09 | INFO     | vitagraph.bio_signal_processor | Processing 'hrv_ms' (300 rows, window=5)


2026-07-18 13:23:09 | INFO     | vitagraph.bio_signal_processor | Processing 'sleep_hours' (30 rows, window=5)


2026-07-18 13:23:09 | INFO     | vitagraph.pipeline | Built knowledge graph for P001 (632 total nodes so far)


...

2026-07-18 13:23:10 | INFO     | vitagraph.knowledge_graph | Exported graph to outputs/notebook_pipeline/knowledge_graph.json (JSON, 1899 nodes)


2026-07-18 13:23:10 | INFO     | vitagraph.knowledge_graph | Exported graph to outputs/notebook_pipeline/knowledge_graph.graphml (GraphML, 1899 nodes)


2026-07-18 13:23:10 | INFO     | vitagraph.pipeline | Pipeline outputs written to outputs/notebook_pipeline


2026-07-18 13:23:10 | INFO     | vitagraph.pipeline | Pipeline complete.


,person_id,chronological_age,heart_rate_avg,hrv_avg,sleep_hours_avg,activity_level,environmental_exposure,predicted_biological_age,age_gap
0,P001,51,72.603502,54.864978,7.338524,0.660943,0.426482,53.01,2.01
1,P002,42,72.516733,54.996880,7.683135,0.750090,0.338482,41.81,-0.19
2,P003,33,72.523333,53.388869,7.049231,0.788113,0.375031,34.41,1.41


In [5]:
result.test_metrics

EvaluationMetrics(mae=2.4103, rmse=2.8991, r2=0.9509)